## Use Python Runtime 3.11 not 3.12

In [1]:
import polars as pl
from datetime import date
import sempy.fabric as fabric

In [2]:
import time
import notebookutils

LAKEHOUSE_NAME = "ecommerce_lakehouse"


def lakehouse_exists(name):
    try:
        notebookutils.lakehouse.get(name)
        return True
    except Exception:
        return False

try:
    if not lakehouse_exists(LAKEHOUSE_NAME):
        print(f"Creating lakehouse '{LAKEHOUSE_NAME}'...")
        notebookutils.lakehouse.create(
            LAKEHOUSE_NAME,
            "lakehouse with ecommerce data for data agent workshop",
            {"enableSchemas": True},
        )
        for _ in range(20):
            try:
                props = notebookutils.lakehouse.getWithProperties(LAKEHOUSE_NAME)["properties"]
                if props.get("sqlEndpointProperties", {}).get("connectionString"):
                    break
            except Exception:
                pass
            time.sleep(5)
        print(f"Lakehouse '{LAKEHOUSE_NAME}' created.")
    else:
        print(f"Lakehouse '{LAKEHOUSE_NAME}' already exists.")

    props = notebookutils.lakehouse.getWithProperties(LAKEHOUSE_NAME)["properties"]
    LAKEHOUSE_ABFSS = f"{props['abfsPath']}/Tables/dbo/"
    LAKEHOUSE_SQLEP = props["sqlEndpointProperties"]["connectionString"]
    LAKEHOUSE_FILES = f"{props['abfsPath']}/Files/"

except Exception as e:
    print(f"Error while creating or retrieving lakehouse: {e}")

Lakehouse 'ecommerce_lakehouse' already exists.


In [3]:
print(LAKEHOUSE_ABFSS)
print(LAKEHOUSE_FILES)
print(LAKEHOUSE_SQLEP)
DATA_PATH = f"{LAKEHOUSE_FILES}Data/"

abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Tables/dbo/
abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Files/
xpkymsttihxelp6vuo2w5ywt2u-vaokmda6cy3ubjpxk5x5mqssqm.datawarehouse.fabric.microsoft.com


In [4]:
import requests
import notebookutils

BASE_URL = "https://raw.githubusercontent.com/pawarbi/ecommerce-data/main/data"
FILES = [
    "customers.parquet",
    "geolocation.parquet",
    "order_items.parquet",
    "order_payments.parquet",
    "order_reviews.parquet",
    "orders.parquet",
    "product_category_name_translation.parquet",
    "products.parquet",
    "sellers.parquet",
    "readme.md",
]


def download_raw_data(target_path):
    notebookutils.fs.mkdirs(target_path)

    existing = {f.name for f in notebookutils.fs.ls(target_path)}

    downloaded, skipped = 0, 0
    for fname in FILES:
        if fname in existing:
            skipped += 1
            continue

        tmp = f"/tmp/{fname}"
        r = requests.get(f"{BASE_URL}/{fname}")
        r.raise_for_status()
        with open(tmp, "wb") as f:
            f.write(r.content)

        notebookutils.fs.cp(f"file://{tmp}", f"{target_path}/{fname}", True)
        downloaded += 1

    return {"path": target_path, "downloaded": downloaded, "skipped": skipped}

download_raw_data(f"{LAKEHOUSE_FILES}Data")

{'path': 'abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Files/Data',
 'downloaded': 0,
 'skipped': 10}

In [5]:
notebookutils.fs.ls(f"{LAKEHOUSE_FILES}Data/")

[FileInfo(path=abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Files/Data/customers.parquet, name=customers.parquet, size=3988213, isDir=False, modifyTime=1778342586000),
 FileInfo(path=abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Files/Data/geolocation.parquet, name=geolocation.parquet, size=13882173, isDir=False, modifyTime=1778342589000),
 FileInfo(path=abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Files/Data/order_items.parquet, name=order_items.parquet, size=4104383, isDir=False, modifyTime=1778342591000),
 FileInfo(path=abfss://0ca61ca8-161e-4037-a5f7-576fd6425283@onelake.dfs.fabric.microsoft.com/dab5014b-6670-4aac-aa58-c76c9452f7c9/Files/Data/order_payments.parquet, name=order_payments.parquet, size=2229657, isDir=False, modifyTime=1778342593000),
 FileInfo(path=abfss://0ca61c

In [7]:
customers = pl.read_parquet(f"{DATA_PATH}customers.parquet")

print(f"Customers: {customers.shape[0]:,} rows")
print(customers.schema)

customers.write_delta(f"{LAKEHOUSE_ABFSS}customers", mode="overwrite")

Customers: 99,441 rows
Schema([('customer_id', String), ('customer_unique_id', String), ('customer_zip_code_prefix', Int32), ('customer_city', String), ('customer_state', String)])


In [8]:
geolocation = pl.read_parquet(f"{DATA_PATH}geolocation.parquet")

print(f"Geolocation: {geolocation.shape[0]:,} rows")
print(geolocation.schema)

geolocation.write_delta(f"{LAKEHOUSE_ABFSS}geolocation", mode="overwrite")

Geolocation: 1,000,163 rows
Schema([('geolocation_zip_code_prefix', Int32), ('geolocation_lat', Float64), ('geolocation_lng', Float64), ('geolocation_city', String), ('geolocation_state', String)])


In [9]:
order_payments = pl.read_parquet(f"{DATA_PATH}order_payments.parquet")

print(f"Order Payments: {order_payments.shape[0]:,} rows")
print(order_payments.schema)

order_payments.write_delta(f"{LAKEHOUSE_ABFSS}order_payments", mode="overwrite")

Order Payments: 103,886 rows
Schema([('order_id', String), ('payment_sequential', Int32), ('payment_type', String), ('payment_installments', Int32), ('payment_value', Float64)])


In [10]:
orders = pl.read_parquet(f"{DATA_PATH}orders.parquet")

print(f"Orders: {orders.shape[0]:,} rows")
print(orders.schema)

orders.write_delta(f"{LAKEHOUSE_ABFSS}orders", mode="overwrite")

Orders: 99,441 rows
Schema([('order_id', String), ('customer_id', String), ('order_status', String), ('order_purchase_timestamp', Datetime(time_unit='ns', time_zone='UTC')), ('order_approved_at', Datetime(time_unit='ns', time_zone='UTC')), ('order_delivered_carrier_date', Datetime(time_unit='ns', time_zone='UTC')), ('order_delivered_customer_date', Datetime(time_unit='ns', time_zone='UTC')), ('order_estimated_delivery_date', Datetime(time_unit='ns', time_zone='UTC'))])


In [11]:
products = pl.read_parquet(f"{DATA_PATH}products.parquet")

print(f"Products: {products.shape[0]:,} rows")
print(products.schema)

products.write_delta(f"{LAKEHOUSE_ABFSS}products", mode="overwrite")

Products: 32,951 rows
Schema([('product_id', String), ('product_category_name', String), ('product_name_lenght', Int32), ('product_description_lenght', Int32), ('product_photos_qty', Int32), ('product_weight_g', Int32), ('product_length_cm', Int32), ('product_height_cm', Int32), ('product_width_cm', Int32)])


In [12]:
sellers = pl.read_parquet(f"{DATA_PATH}sellers.parquet")

print(f"Sellers: {sellers.shape[0]:,} rows")
print(sellers.schema)

sellers.write_delta(f"{LAKEHOUSE_ABFSS}sellers", mode="overwrite")

Sellers: 3,095 rows
Schema([('seller_id', String), ('seller_zip_code_prefix', Int32), ('seller_city', String), ('seller_state', String)])


In [13]:
product_category_translation = pl.read_parquet(f"{DATA_PATH}product_category_name_translation.parquet")

print(f"Product Category Translation: {product_category_translation.shape[0]:,} rows")
print(product_category_translation.schema)

product_category_translation.write_delta(f"{LAKEHOUSE_ABFSS}product_category_translation", mode="overwrite")

Product Category Translation: 71 rows
Schema([('product_category_name', String), ('product_category_name_english', String)])


In [14]:
order_items = pl.read_parquet(f"{DATA_PATH}order_items.parquet")

print(f"Order Items: {order_items.shape[0]:,} rows")
print(order_items.schema)

order_items.write_delta(f"{LAKEHOUSE_ABFSS}order_items", mode="overwrite")

Order Items: 112,650 rows
Schema([('order_id', String), ('order_item_id', Int32), ('product_id', String), ('seller_id', String), ('shipping_limit_date', Datetime(time_unit='ns', time_zone='UTC')), ('price', Float64), ('freight_value', Float64)])


In [15]:
def create_dim_date(start_date: date = date(2016, 1, 1), end_date: date = date(2020, 12, 31)) -> pl.DataFrame:
    """Create date dimension table"""
    
    date_range = pl.date_range(start_date, end_date, eager=True).alias("date")
    
    df = pl.DataFrame({"date": date_range}).with_columns([
        pl.col("date").dt.strftime("%Y%m%d").cast(pl.Int32).alias("date_key"),
        pl.col("date").dt.year().cast(pl.Int16).alias("year"),
        pl.col("date").dt.month().cast(pl.Int8).alias("month_number"),
        pl.col("date").dt.strftime("%B").alias("month_name"),
        pl.col("date").dt.quarter().cast(pl.Int8).alias("quarter"),
        (pl.col("date").dt.weekday() + 1).cast(pl.Int8).alias("day_of_week"),
        pl.col("date").dt.strftime("%A").alias("day_name"),
        pl.col("date").dt.strftime("%Y-%m").alias("year_month"),
        pl.col("date").dt.day().cast(pl.Int8).alias("day_of_month"),
        pl.col("date").dt.ordinal_day().cast(pl.Int16).alias("day_of_year"),
        (pl.col("date").dt.weekday() >= 5).alias("is_weekend")
    ])
    
    return df.select([
        "date_key", "date", "year", "quarter", "month_number", "month_name",
        "year_month", "day_of_month", "day_of_week", "day_name", 
        "day_of_year", "is_weekend"
    ])

dim_date = create_dim_date()
print(f"Date Dimension: {dim_date.shape[0]:,} rows")
print(dim_date.schema)


# In[ ]:


dim_date.write_delta(f"{LAKEHOUSE_ABFSS}dim_date", mode="overwrite")

Date Dimension: 1,827 rows
Schema([('date_key', Int32), ('date', Date), ('year', Int16), ('quarter', Int8), ('month_number', Int8), ('month_name', String), ('year_month', String), ('day_of_month', Int8), ('day_of_week', Int8), ('day_name', String), ('day_of_year', Int16), ('is_weekend', Boolean)])
